# Revisión capa GOLD (marts Power BI + feature stores ML)

Los 6 marts para dashboards usan **grano agregado** (resúmenes por
fecha/hora/zona/servicio); el detalle viaje a viaje vive en silver/star/facts.
**Prueba de no-pérdida**: `SUM(viajes)` de cada mart == filas de los facts, exacto.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Inventario de salidas gold

In [2]:
inventario = []
for sub in ("marts", "ml"):
    base = DATA / "gold" / sub
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        inventario.append({"tipo": sub, "tabla": d.name, "filas": parquet_rows(d), "peso_mb": dir_mb(d)})
inv = pl.DataFrame(inventario)
print(f"gold total: {inv['peso_mb'].sum():,.0f} MB")
inv

gold total: 3,535 MB


tipo,tabla,filas,peso_mb
str,str,i64,f64
"""marts""","""mart_abc_xyz_zones""",2570,0.13
"""marts""","""mart_demand_volume""",19855078,150.56
"""marts""","""mart_financial_performance""",2569006,88.14
"""marts""","""mart_operational_profile""",2569006,51.33
"""marts""","""mart_supply_demand_balance""",24951595,154.49
"""marts""","""mart_tipping_behavior""",462060,6.6
"""ml""","""kmodes_model""",299773,3.23
"""ml""","""ml_feat_arima_trips""",565088,2.47
"""ml""","""ml_feat_isolation_fraud""",113645848,1768.27


## Cuadre de conteos: cada viaje contado exactamente una vez

financial/operational/tipping excluyen `fhv` **por diseño** (esa categoría
no publica tarifas ni distancias), por eso suman 292M y no 317M.

In [3]:
MART_CATS = {
    "mart_demand_volume": CATS,
    "mart_financial_performance": ["green", "yellow", "fhvhv"],
    "mart_operational_profile": ["green", "yellow", "fhvhv"],
    "mart_tipping_behavior": ["green", "yellow", "fhvhv"],
}
cuadre = []
for mart, cats in MART_CATS.items():
    suma = (pl.scan_parquet(str(DATA / "gold" / "marts" / mart / "**" / "*.parquet"), hive_partitioning=True)
              .select(pl.col("viajes").sum()).collect().item())
    facts = sum(parquet_rows(DATA / "silver" / "star" / "facts" / f"fact_{c}_trip") or 0 for c in cats)
    cuadre.append({"mart": mart, "sum_viajes": suma, "filas_facts": facts, "cuadra": suma == facts})
pl.DataFrame(cuadre)

mart,sum_viajes,filas_facts,cuadra
str,i64,i64,bool
"""mart_demand_volume""",840502449,840502449,true
"""mart_financial_performance""",829157485,829157485,true
"""mart_operational_profile""",829157485,829157485,true
"""mart_tipping_behavior""",829157485,829157485,true


## Vista previa de cada mart

In [4]:
for mart in sorted((DATA / "gold" / "marts").iterdir()):
    if not mart.is_dir():
        continue
    df = pl.read_parquet(str(mart / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {mart.name} ({len(df.columns)} columnas) ===")
    print(df.head(3))


=== mart_abc_xyz_zones (12 columnas) ===
shape: (3, 12)
┌────────┬────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬──────┐
│ pu_loc ┆ boroug ┆ zone   ┆ ingres ┆ viaje ┆ viaje ┆ coefi ┆ clase ┆ porce ┆ clase ┆ servi ┆ year │
│ ation_ ┆ h      ┆ ---    ┆ os_tot ┆ s_dia ┆ s_dia ┆ cient ┆ _xyz  ┆ ntaje ┆ _abc  ┆ ce_id ┆ ---  │
│ id     ┆ ---    ┆ str    ┆ ales_z ┆ rios_ ┆ rios_ ┆ e_var ┆ ---   ┆ _acum ┆ ---   ┆ ---   ┆ i64  │
│ ---    ┆ str    ┆        ┆ ona    ┆ prome ┆ std   ┆ iacio ┆ str   ┆ ulado ┆ str   ┆ str   ┆      │
│ f64    ┆        ┆        ┆ ---    ┆ dio   ┆ ---   ┆ n_xyz ┆       ┆ _ingr ┆       ┆       ┆      │
│        ┆        ┆        ┆ f64    ┆ ---   ┆ f64   ┆ ---   ┆       ┆ esos  ┆       ┆       ┆      │
│        ┆        ┆        ┆        ┆ f64   ┆       ┆ f64   ┆       ┆ ---   ┆       ┆       ┆      │
│        ┆        ┆        ┆        ┆       ┆       ┆       ┆       ┆ f64   ┆       ┆       ┆      │
╞════════╪════════╪════════╪══════

## Análisis de ejemplo 1: demanda mensual por servicio (millones de viajes)

In [5]:
demanda = (pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                           hive_partitioning=True)
    .group_by("service_id", "month").agg((pl.col("viajes").sum() / 1e6).round(2).alias("millones"))
    .collect()
    .pivot(values="millones", index="month", on="service_id")
    .sort("month"))
demanda

month,green,yellow,fhvhv,fhv
i64,f64,f64,f64,f64
1,0.16,8.63,58.55,0.81
2,0.15,10.66,56.66,0.73
3,0.17,9.56,62.23,0.81
4,0.16,9.38,58.63,1.01
5,0.18,9.97,61.64,1.03
6,0.16,9.29,59.36,1.05
7,0.15,8.32,57.97,1.05
8,0.15,7.98,56.72,0.95
9,0.16,8.88,58.5,1.13


## Análisis de ejemplo 2: top 10 zonas por demanda anual

In [6]:
(pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                 hive_partitioning=True)
   .group_by("pu_zone", "pu_borough").agg(pl.col("viajes").sum().alias("viajes"))
   .sort("viajes", descending=True).head(10).collect())

pu_zone,pu_borough,viajes
str,str,i64
"""JFK Airport""","""Queens""",18820984
"""LaGuardia Airport""","""Queens""",18049537
"""Midtown Center""","""Manhattan""",14003405
"""Times Sq/Theatre District""","""Manhattan""",12693978
"""East Village""","""Manhattan""",11924792
"""Upper East Side South""","""Manhattan""",11465504
"""East Chelsea""","""Manhattan""",11004579
"""Union Sq""","""Manhattan""",10707759
"""Midtown South""","""Manhattan""",10132490


## Feature stores ML

- `ml_feat_isolation_fraud`: yellow+green **completos** (el modelo puntúa fraude viaje a viaje).
- `ml_feat_kmodes_trips`: yellow+green completos + **muestra 5% de fhvhv** (el modelo K-Modes entrena con máx. 100k filas por servicio; la muestra es 120× eso).
- `ml_feat_arima_trips`: serie agregada zona×hora para pronóstico.

In [7]:
# Preview solo de los 3 FEATURE STORES (ml_feat_*). Los directorios de
# salidas de modelos (--gold-ml: kmodes_model, ml_isolation_fraud_scores,
# ml_sarimax_trips_forecast) usan otros esquemas de particion y se listan aparte.
for store in sorted((DATA / "gold" / "ml").glob("ml_feat_*")):
    if not store.is_dir():
        continue
    df = pl.read_parquet(str(store / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {store.name}: {parquet_rows(store):,} filas | {dir_mb(store):,.0f} MB ===")
    print(df.head(3))

print("\n--- salidas de modelos entrenados (--gold-ml) ---")
for d in sorted((DATA / "gold" / "ml").iterdir()):
    if d.is_dir() and not d.name.startswith("ml_feat_"):
        n = parquet_rows(d)
        print(f"{d.name}: {n:,} filas | {dir_mb(d):,.1f} MB" if n else f"{d.name}: {dir_mb(d):,.1f} MB")


=== ml_feat_arima_trips: 565,088 filas | 2 MB ===
shape: (3, 6)
┌────────────┬─────────────────────┬────────────┬─────────┬──────┬───────┐
│ service_id ┆ pickup_hour         ┆ trip_count ┆ borough ┆ year ┆ month │
│ ---        ┆ ---                 ┆ ---        ┆ ---     ┆ ---  ┆ ---   │
│ str        ┆ datetime[ns]        ┆ i64        ┆ str     ┆ i64  ┆ i64   │
╞════════════╪═════════════════════╪════════════╪═════════╪══════╪═══════╡
│ green      ┆ 2023-01-17 20:00:00 ┆ 1          ┆ Bronx   ┆ 2023 ┆ 1     │
│ green      ┆ 2023-01-12 16:00:00 ┆ 1          ┆ Bronx   ┆ 2023 ┆ 1     │
│ green      ┆ 2023-01-08 08:00:00 ┆ 1          ┆ Bronx   ┆ 2023 ┆ 1     │
└────────────┴─────────────────────┴────────────┴─────────┴──────┴───────┘

=== ml_feat_isolation_fraud: 113,645,848 filas | 1,768 MB ===
shape: (3, 14)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬────────┬───────┬───────┬───────┬──────┬───────┐
│ tri ┆ rat ┆ dur ┆ tri ┆ far ┆ ext ┆ mta ┆ veloci ┆ costo_ ┆ ratio ┆ is_an ┆ ser